# データベース演習 第11回

### 🔍 利用方法

- このノートブックは**閲覧専用**です。  
  自分のGoogleドライブにコピーを作成してから編集してください。

  1. メニューの「**ファイル → ドライブにコピーを保存**」を選ぶ  
  2. コピーしたノートブックの**ファイル名（左上の"xxxx"）を学籍番号に変更**してください（提出時の識別のため）

---

### 🔧 提出手順

1. Google Colab の右上にある「**共有**」ボタンをクリック  
2. 「**一般的なアクセス**」の設定を「**リンクを知っている全員**」に変更  
3. アクセス権を「**閲覧者**」に設定（⚠️「編集者」にしないこと！）  
4. 表示されるURLをコピー  
5. WebClassの提出フォームに、その**URLを貼り付けて提出**

---

### 💡 補足・注意点

- **「編集者」ではなく「閲覧者」**に設定してください。  
  → 教員が間違ってファイルを上書きしてしまうのを防ぐためです。
- **提出したリンクを自分でも一度開いてみて、ちゃんと共有されているかを確認**してください。

### 使用するデータベースのダウンロード

* 今回は，サイズの大きなweblogデータベースを使用します（ファイル名：weblog.sqlite3）．
* 第11回よりaccess_log_wideを含むすべてのテーブルが含まれています．
* これまでは，githubからダウンロードしていましたが，サイズの関係でGoogle Driveからダウンロードします．
* Colabのランタイムが切り替わると，その度にダウンロードする必要があります
* その度にダウンロードするのが面倒な人は，自分のGoogleドライブに保存し，そのファイルを参照する方法もあります（生成AIなどで調べてみてください）．

### データベースをダウンロード

In [ ]:
# Google DriveからSQLiteファイルを取得
import gdown

# ファイルIDを指定
file_id = '1nuCOXMd3H0rL-82D460SnbpM7YjLNBeq'
url = f'https://drive.google.com/uc?id={file_id}'

# 保存ファイル名を指定
# 第11回よりaccess_log_wideを含むすべてのテーブルが含まれています
output = 'weblog.sqlite3'
gdown.download(url, output, quiet=False)

### JupySQLのインストールと有効化

In [ ]:
# Colabではjupysqlのインストールが必要
import sys
if 'google.colab' in sys.modules:
    %pip install -q jupysql

In [ ]:
# jupysqlの拡張機能を有効化`
%load_ext sql

In [ ]:
# 結果表示数は以下の数値を変えれば変更できる
%config SqlMagic.displaylimit = 10 # デフォルトは10行

### Webログデータベース

In [ ]:
# Webログデータベースに接続する
%sql sqlite:///weblog.sqlite3

### 含まれるテーブルの確認

In [ ]:
%%sql
SELECT * FROM sqlite_master;


## 例題1

customer_locationsは，顧客の居住地域（都道府県）を表すテーブルである（データは3顧客のみ）．アクセスの要求時間とアクセスした人の居住地域の対応関係を知るために，access_logテーブルにcustomer_locationsテーブルをcustomer_idが等しいことによって左側外部結合し，request_timeとprefectureの組を求めよ（顧客登録していないアクセスした人のログ（= customer_idがnull）を除外しないために外部結合を使う）．ただし，結果のデータ数を制限するために以下の指定を行うこと

* request_pathが/searchに等しいものに限定すること
* 結果は，prefectureによってソートすること（PostgreSQL→昇順，SQLite→降順とせよ．NULLが最初か最後か違うため）
* 最大100個の出力制限を付けること

In [ ]:
%%sql
SELECT a.request_time, l.prefecture FROM access_log AS a
LEFT OUTER JOIN customer_locations AS l
ON a.customer_id = l.customer_id
WHERE a.request_path = '/search'
ORDER BY l.prefecture DESC
LIMIT 100;

## 例題2

itemsテーブルは，商品の番号（item_id），商品の名前（item_name），商品の価格（item_price），商品を販売する店舗の番号（shop_id）のカラムを持つ．このitemsテーブルをセルフジョインすることにより，商品のすべての組み合わせを作ることができる．さらに，**同じ店舗で購入できる**商品の組み合わせ（商品1・商品2と呼ぶ）を求めたい．そこで，店舗の番号，商品1の名前（item_name1），商品2の名前（item_name2），2つの商品の合計価格（total_price）を求めるSQL文を作成せよ．次ページに結果の例を示す

注意
* 組合せには同じ商品の重複と，順序入れ替えの重複は許すこととする
  (練習として重複を排除する方法を考えてみよう→次節に関連)

In [ ]:
%%sql
SELECT i1.shop_id, i1.item_name AS item_name1, i2.item_name AS item_name2,
    i1.item_price + i2.item_price AS total_price
FROM items AS i1, items AS i2
WHERE i1.shop_id = i2.shop_id;

参考：商品の組合せの重複を排除

In [ ]:
%%sql
SELECT i1.shop_id, i1.item_name AS item_name1, i2.item_name AS item_name2,
    i1.item_price + i2.item_price AS total_price
FROM items AS i1, items AS i2
WHERE i1.shop_id = i2.shop_id
AND i1.item_id > i2.item_id;

## 演習 課題5-3

access_logテーブルとcustomer_locationsテーブルを用いて，要求されたURL（request_path）と居住地域別のアクセス数の集計を求めたい．また，例題と同様に，居住地域が登録されていない場合の集計結果も含めたい（結果例を下図に示す）．例題を参考に，この集計を行うSQL文を作成せよ．ただし，検索結果は10個のみに制限すること


## 演習 課題5-4

前に述べた「1年前との売上比較」において，前年から当年への売上額の増減額も一緒にみたいという要望があった．増減額を計算した結果も併せて表示するSQL文を作成せよ．このカラムの別名はsales_diffとせよ．以下に結果の例を示す


## 演習 課題5-5

前ページのSQL文の結果に対して，注文の多かった商品の組合せを知りたいという要望があった．そこで，商品の組合せに対する注文数が多い順から10行出力するようなSQL文を作成せよ．結果例を以下に示す